In [1]:
import pandas as pd
from pathlib import Path

In [2]:
DATASET_PATH = Path("../datasets/dataset.csv")

Load Dataset

In [3]:
dataset = pd.read_csv(DATASET_PATH)

Check all datatypes

In [4]:
print(dataset.dtypes)


property_id             str
date                    str
property_type           str
location_score        int64
amenities_score       int64
rating              float64
base_price          float64
month                 int64
weekday                 str
weekend               int64
season                  str
holiday               int64
demand              float64
competitor_price    float64
nearby_event          int64
market_trend        float64
final_price         float64
occupancy_rate      float64
revenue             float64
dtype: object


Convert Date type from `str` to `datetime64[us]`

In [5]:
dataset['date'] = pd.to_datetime(dataset['date'])
print(dataset.dtypes)

property_id                    str
date                datetime64[us]
property_type                  str
location_score               int64
amenities_score              int64
rating                     float64
base_price                 float64
month                        int64
weekday                        str
weekend                      int64
season                         str
holiday                      int64
demand                     float64
competitor_price           float64
nearby_event                 int64
market_trend               float64
final_price                float64
occupancy_rate             float64
revenue                    float64
dtype: object


Check for NA values

In [6]:
na_values = dataset.isna()
print(f"Number of NA values: {len(na_values[na_values.eq(True).any(axis='columns')])}")

null_values = dataset.isnull()
print(f"Number of NULL values: {len(null_values[null_values.eq(True).any(axis='columns')])}")

Number of NA values: 0
Number of NULL values: 0


One-hot encode all the categorical features

In [7]:
dataset = pd.get_dummies(
  dataset,
  columns=['season'],
  dtype=int
)

dataset = pd.get_dummies(
  dataset,
  columns=['weekday'],
  dtype=int
)

dataset = pd.get_dummies(
  dataset,
  columns=['property_type'],
  dtype=int
)

Add lag features to enrich the dataset and to get better model prediction accuracy

In [8]:
dataset['occupancy_lag_1'] = (
    dataset
    .groupby('property_id')['demand']
    .shift(1)
)

dataset['price_lag_1'] = (
  dataset
  .groupby('property_id')['final_price']
  .shift(1)
)

dataset['demand_lag_1'] = (
  dataset
  .groupby('property_id')['demand']
  .shift(1)
)
dataset = dataset.sort_values(
    ['property_id', 'date']
)

dataset['rolling_7_day_demand'] = (
    dataset
    .groupby('property_id')['demand']
    .transform(
        lambda x:
        x.rolling(7, min_periods=1).mean()
    )
)

dataset['rolling_30_day_price'] = (
    dataset
    .groupby('property_id')['final_price']
    .transform(
        lambda x:
           x.rolling(window=30,min_periods=1).mean() 
    )    
)

Save the final processed dataset

In [9]:
dataset.to_csv("../datasets/processed_dataset.csv",index=False)

Split the dataset into Training and Testing datasets

In [10]:
TRAIN_SPLIT = int(0.8 * len(dataset))
TEST_SPLIT = int(0.2 * len(dataset))

train_dataset = (
  dataset.groupby(['property_id']).apply(lambda x: x[:int(0.8 * len(x))])
)

test_dataset = (
  dataset.groupby('property_id').apply(lambda x: x[int(0.8 * len(x)):])
)

train_dataset.to_csv('../datasets/train_dataset.csv')
test_dataset.to_csv('../datasets/test_dataset.csv')